# Glance プロンプトチューニング環境

このNotebookでは、InternVL 3.5 GGUFモデルを使用してプロンプトの効果を確認できます。

## 使い方
1. セル1-3を順に実行してモデルをロード
2. セル4（プロンプトチューニング）のプロンプトを変更して何度も実行
3. 良いプロンプトが見つかったら `config.yaml` に反映

## セル1: 環境セットアップ

In [1]:
import sys
import os
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import yaml
from typing import Dict, Any
import warnings
warnings.filterwarnings('ignore')

# プロジェクトパスを設定
project_root = Path('.').resolve()
# models_dir = project_root / 'models'
models_dir = project_root
test_images_dir = project_root / 'test_images'
config_file = project_root / 'config.yaml'

print(f"📁 プロジェクトパス: {project_root}")
print(f"📁 モデルディレクトリ: {models_dir}")
print(f"📁 テスト画像ディレクトリ: {test_images_dir}")

# モデルパスの確認
with open(config_file, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
    active_model = config['activeModel']
    model_config = config['models'][active_model]
    
    model_path = models_dir / model_config['path']
    mmproj_path = models_dir / model_config['mmproj_path']
    
print(f"\n📦 アクティブモデル: {active_model}")
print(f"   モデルパス: {model_path}")
print(f"   モデル存在: {model_path.exists()}")
print(f"   ビジョンプロジェクタ存在: {mmproj_path.exists()}")

📁 プロジェクトパス: /Users/takeshi/Project/Glance/glance-pyapp/python-backend
📁 モデルディレクトリ: /Users/takeshi/Project/Glance/glance-pyapp/python-backend
📁 テスト画像ディレクトリ: /Users/takeshi/Project/Glance/glance-pyapp/python-backend/test_images

📦 アクティブモデル: qwen3-vl-4b-server
   モデルパス: /Users/takeshi/Project/Glance/glance-pyapp/python-backend/models/gguf/Qwen3VL-4B-Instruct-Q4_K_M.gguf
   モデル存在: True
   ビジョンプロジェクタ存在: True


## セル2: モデルのロード

In [2]:
# # モデルのインポート
# sys.path.insert(0, str(project_root))
# from models.internvl_gguf import InternVLGGUFModel

# # モデルのインスタンス作成とロード
# print("🚀 モデルをロード中...（初回は数分かかります）\n")
# model = InternVLGGUFModel(
#     model_path=str(model_path),
#     mmproj_path=str(mmproj_path)
# )
# model.load()

# print(f"\n✅ モデルのロード完了")
# print(f"   情報: {model.get_info()}")

# モデルのインポート
sys.path.insert(0, str(project_root))
from models.qwen3_vl_server import Qwen3VLServerModel

# llama-server への接続設定
server_url = "http://127.0.0.1:8080"

# モデルのインスタンス作成とロード
print("🚀 llama-server への接続を確認中...\n")
model = Qwen3VLServerModel(
    model_path=str(model_path),
    mmproj_path=str(mmproj_path),
    server_url=server_url
)
model.load()

print(f"\n✅ llama-server への接続完了")
print(f"   情報: {model.get_info()}")


🚀 llama-server への接続を確認中...

📦 Qwen3-VL Server接続確認中: http://127.0.0.1:8080
✅ llama-server 接続成功

✅ llama-server への接続完了
   情報: {'name': 'Qwen3-VL (llama-server)', 'path': '/Users/takeshi/Project/Glance/glance-pyapp/python-backend/models/gguf/Qwen3VL-4B-Instruct-Q4_K_M.gguf', 'mmproj_path': '/Users/takeshi/Project/Glance/glance-pyapp/python-backend/models/gguf/mmproj-Qwen3VL-4B-Instruct-Q8_0.gguf', 'is_loaded': True, 'server_url': 'http://127.0.0.1:8080', 'type': 'qwen3_vl_server'}


## セル3: ユーティリティ関数とテスト画像の準備

In [14]:
def load_test_images():
    """test_imagesディレクトリから画像を読み込む"""
    images = {}
    if not test_images_dir.exists():
        print(f"⚠️  テスト画像ディレクトリが見つかりません: {test_images_dir}")
        return images
    
    for img_path in sorted(test_images_dir.glob('*.png')) + sorted(test_images_dir.glob('*.jpg')) + sorted(test_images_dir.glob('*.jpeg')):
        try:
            images[img_path.stem] = Image.open(img_path)
            print(f"✅ 読み込み: {img_path.name}")
        except Exception as e:
            print(f"❌ エラー: {img_path.name} - {e}")
    
    return images

def run_inference(image: Image.Image, prompt: str, **kwargs) -> str:
    """推論を実行"""
    try:
        result = model.inference(image, prompt, **kwargs)
        return result
    except Exception as e:
        return f"エラー: {str(e)}"

def display_result(prompt_name: str, result: str, max_length: int = 500):
    """結果を表示"""
    print(f"\n{'='*60}")
    print(f"プロンプト: {prompt_name}")
    print(f"{'='*60}")
    if len(result) > max_length:
        print(result[:max_length] + f"\n... (全{len(result)}文字)")
    else:
        print(result)
    print()

# テスト画像を読み込む
print("📸 テスト画像を読み込み中...\n")
test_images = load_test_images()

if test_images:
    print(f"\n✅ {len(test_images)}個の画像を読み込みました")
    # 最初の画像を表示
    first_image_name = list(test_images.keys())[3]
    first_image = test_images[first_image_name]
    print(f"\n使用するテスト画像: {first_image_name}")
    print(f"サイズ: {first_image.size}")
else:
    print(f"\n⚠️  test_imagesディレクトリにまず画像を配置してください")
    print(f"パス: {test_images_dir}")

📸 テスト画像を読み込み中...

✅ 読み込み: Google検索画面.png
✅ 読み込み: Google画面.png
✅ 読み込み: Teams画面.png
✅ 読み込み: VSCode編集画面.png
✅ 読み込み: グラフ(総人口).png
✅ 読み込み: デスクトップ画面.png
✅ 読み込み: デスクトップ画面2.jpg

✅ 7個の画像を読み込みました

使用するテスト画像: VSCode編集画面
サイズ: (2934, 1842)


## セル4: プロンプトチューニング（★メイン）

このセルを何度も実行してプロンプトを調整できます。

In [15]:
# 選択するテスト画像
if test_images:
    image_name = first_image_name  # または list(test_images.keys())[0] で別の画像を選択
    test_image = test_images[image_name]
    
    # ===== ここからプロンプトを編集 =====
    
    # 【パターン1】標準的なプロンプト
    prompt_1 = """画面に表示されている内容を20文字程度で簡潔に要約してください。
図や表がある場合は、そのタイトルや数値、傾向を読み取って説明に含めてください。
全体が把握できない場合でも、認識できた要素（テキストやボタンなど）を列挙してください。
ここでは詳細を述べる必要はありません。詳細のフェーズは別にあるので、あくまで概要を把握できる簡素な説明を提供してください。"""
    
    # ===== ここまでプロンプトを編集 =====
    
    # 推論パラメータ
    inference_params = {
        'temperature': 0.1,    # 低いほど安定した出力
        'max_tokens': 200,     # 最大出力トークン数
        'top_p': 0.9,
        'repetition_penalty': 1.15,  # 繰り返しにペナルティ（1.0-2.0、デフォルト1.0）
        'no_repeat_ngram_size': 3,  # 同じN-gramの繰り返しを防ぐ
        'max_token': 200  # モデルの最大トークン数
    }
    
    print(f"🖼️  テスト画像: {image_name}")
    print(f"🔧 パラメータ: temperature={inference_params['temperature']}, max_tokens={inference_params['max_tokens']}\n")
    
    # 複数のプロンプトを実行
    results = {}
    
    print("⏳ パターン1を推論中...")
    results['パターン1'] = run_inference(test_image, prompt_1, **inference_params)
    display_result('パターン1', results['パターン1'])

🖼️  テスト画像: VSCode編集画面
🔧 パラメータ: temperature=0.1, max_tokens=200

⏳ パターン1を推論中...
✅ 画像分析完了（37文字）

プロンプト: パターン1
開発者ツールでJavaScriptコードとコンソール出力が表示されている。



## セル5: 詳細分析プロンプトのテスト

In [16]:
if test_images:
    # 詳細分析用のプロンプト
    detailed_prompt = """画面に表示されている内容を、以下の順序で詳細に日本語で説明してください：
    
1. 全体の概要: 何のアプリケーション・画面か（例: Teams会議、Excel、Webブラウザ）
2. 主要な情報: 最も重要な要素（タイトル、見出し、メインコンテンツ）
3. グラフ・図表: 種類、軸ラベル、データの傾向、具体的な数値を詳細に

曖昧な表現は避け、可能な限り正確かつ詳細に説明してください。
数値やテキストは正確に読み上げてください。"""
    
    inference_params_detailed = {
        'temperature': 0.1,
        'max_tokens': 500,     # 詳細分析なので長め
        'top_p': 0.9,
    }
    
    print(f"🖼️  テスト画像: {image_name}")
    print(f"🔧 パラメータ: temperature={inference_params_detailed['temperature']}, max_tokens={inference_params_detailed['max_tokens']}\n")
    
    print("⏳ 詳細分析を推論中...")
    detailed_result = run_inference(test_image, detailed_prompt, **inference_params_detailed)
    display_result('詳細分析プロンプト', detailed_result, max_length=800)

🖼️  テスト画像: VSCode編集画面
🔧 パラメータ: temperature=0.1, max_tokens=500

⏳ 詳細分析を推論中...
✅ 画像分析完了（945文字）

プロンプト: 詳細分析プロンプト
この画面は、開発者向けのデベロッパーツール（おそらくChrome DevToolsや類似のウェブ開発ツール）のスクリーンショットです。具体的には、ウェブアプリケーションの開発・診断・テストを行うためのインターフェースであり、ブラウザのコンソールやソースコード、ネットワークリクエスト、エレメント構造などを表示しています。

---

**1. 全体の概要**

この画面は「ウェブ開発ツール」（DevTools）の画面です。左側には「Elements」タブ（HTML構造の表示）と「Sources」タブ（ソースコードの表示）が見え、右側には「Console」タブ（コンソール出力）と「Network」タブ（ネットワークリクエストログ）が表示されています。これは、ウェブページの開発者やテスト担当者が、HTML、CSS、JavaScript、ネットワーク通信の状態を確認・分析するためのツールです。画面の上部には「DevTools」のタイトルバーがあり、ツールの各セクションを切り替えるためのタブが並んでいます。

---

**2. 主要な情報**

- **左側の「Elements」タブ**：
  - HTML構造のツリーが表示されており、「body」要素の下に「div」要素が存在し、その中に「h1」要素と「p」要素が含まれています。
  - 「h1」要素のテキストは「Hello, World!」です。
  - 「p」要素のテキストは「This is a paragraph.」です。
  - さらに、CSSクラス「container」が適用されている「div」要素が存在し、その内側に「h1」要素と「p」要素が含まれています。

- **右側の「Console」タブ**：
  - コンソール出力が表示されており、JavaScriptの実行結果やエラーログが記録されています。
  - 例：
... (全945文字)



## セル6: 質問応答プロンプトのテスト

In [ ]:
if test_images:
    # 質問応答用のプロンプト
    question = "画面上に表示されているボタンやメニュー項目は何がありますか？"
    
    question_prompt = f"""以下の質問に基づいて、画面に表示されている内容について詳細に回答してください：

質問: {question}

全体が把握できない場合でも、認識できた要素（テキストやボタンなど）を列挙してください。"""
    
    inference_params_question = {
        'temperature': 0.1,
        'max_tokens': 200,
        'top_p': 0.9,
    }
    
    print(f"🖼️  テスト画像: {image_name}")
    print(f"❓ 質問: {question}")
    print(f"🔧 パラメータ: temperature={inference_params_question['temperature']}, max_tokens={inference_params_question['max_tokens']}\n")
    
    print("⏳ 質問応答を推論中...")
    question_result = run_inference(test_image, question_prompt, **inference_params_question)
    display_result('質問応答プロンプト', question_result)

## セル7: 結果の保存

良いプロンプトが見つかったら、ここで config.yaml に反映するコードを実行できます。

In [ ]:
print("✅ プロンプトチューニングが完了しました")
print("\n📝 良いプロンプトが見つかったら、以下の手順で config.yaml に反映してください：")
print(f"\n1. {config_file} を編集")
print("2. prompt セクションの systemPrompt / detailedPrompt / questionPrompt を更新")
print("3. Pythonバックエンドを再起動")
print("\nまたは、以下のコードで自動更新できます：")
print("""
with open(config_file, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# systemPromptを更新
config['prompt']['systemPrompt'] = \"新しいシステムプロンプト\"

with open(config_file, 'w', encoding='utf-8') as f:
    yaml.dump(config, f, ensure_ascii=False, default_flow_style=False, allow_unicode=True)

print("✅ config.yaml を更新しました")
""")

## セル8: クリーンアップ（終了時に実行）

In [ ]:
# モデルをアンロード
print("🛑 モデルをアンロード中...")
model.unload()
print("✅ メモリを解放しました")